# Concurrencia en Go — Ejercicios + Actividad integradora

1. Crear 5 goroutines y esperar con `WaitGroup`
2. Convertir el ejemplo a channels: productor → workers → consumidor
3. Agregar timeout con `context.WithTimeout`
4. Provocar un deadlock y explicar quién espera a quién
5. Ejecutar con `go test -race` / `go run -race` y corregir una carrera
6. **Actividad integradora:** crawler simulado con cancelación a los 2 segundos

> Colab no trae Go instalado, así que primero lo instalamos (tarda ~20 s).

In [ ]:
%%bash
set -e
if [ ! -d /usr/local/go ]; then
  wget -q https://go.dev/dl/go1.22.5.linux-amd64.tar.gz -O /tmp/go.tgz
  tar -C /usr/local -xzf /tmp/go.tgz
fi
/usr/local/go/bin/go version

go version go1.22.5 linux/amd64


In [ ]:
import os
os.environ["PATH"] += ":/usr/local/go/bin"
os.environ["CGO_ENABLED"] = "1"   # necesario para el detector de carreras (-race)
os.environ["GOCACHE"] = "/content/.gocache"

In [ ]:
%%bash
mkdir -p /content/go_ej1
cd /content/go_ej1
mkdir -p 01_waitgroup 02_pipeline 03_timeout 04_deadlock 05_carrera 06_crawler
[ -f go.mod ] || go mod init ejercicio1

go: creating new go.mod: module ejercicio1
go: to add module requirements and sums:
	go mod tidy


## 1) 5 goroutines + `sync.WaitGroup`

In [ ]:
%%writefile /content/go_ej1/01_waitgroup/main.go
package main

import (
	"fmt"
	"sync"
	"time"
)

func main() {
	var wg sync.WaitGroup

	for i := 1; i <= 5; i++ {
		wg.Add(1)
		go func(id int) {
			defer wg.Done()
			time.Sleep(time.Duration(id) * 100 * time.Millisecond)
			fmt.Printf("goroutine %d terminó\n", id)
		}(i) // pasamos i como argumento para no capturar la variable del loop
	}

	wg.Wait()
	fmt.Println("todas las goroutines terminaron")
}

Writing /content/go_ej1/01_waitgroup/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run ./01_waitgroup 2>&1 || echo "--> terminó con error (código $?)"

goroutine 1 terminó
goroutine 2 terminó
goroutine 3 terminó
goroutine 4 terminó
goroutine 5 terminó
todas las goroutines terminaron


## 2) Pipeline con channels: productor → workers → consumidor
- **Productor**: genera números y cierra el channel de jobs cuando termina.
- **Workers (3)**: leen de `jobs`, calculan el cuadrado y escriben en `results`.
- **Consumidor** (`main`): lee `results` hasta que se cierre. Un goroutine auxiliar espera a los workers con `WaitGroup` y recién ahí cierra `results`.

In [ ]:
%%writefile /content/go_ej1/02_pipeline/main.go
package main

import (
	"fmt"
	"sync"
	"time"
)

type Result struct {
	Worker int
	In     int
	Out    int
}

// Productor: emite 1..n y cierra el channel al terminar.
func producer(n int) <-chan int {
	out := make(chan int)
	go func() {
		defer close(out)
		for i := 1; i <= n; i++ {
			out <- i
		}
	}()
	return out
}

// Worker: consume jobs y produce resultados.
func worker(id int, jobs <-chan int, results chan<- Result, wg *sync.WaitGroup) {
	defer wg.Done()
	for n := range jobs {
		time.Sleep(50 * time.Millisecond) // simula trabajo
		results <- Result{Worker: id, In: n, Out: n * n}
	}
}

func main() {
	jobs := producer(10)
	results := make(chan Result)

	var wg sync.WaitGroup
	for w := 1; w <= 3; w++ {
		wg.Add(1)
		go worker(w, jobs, results, &wg)
	}

	// Cierra results cuando todos los workers terminaron.
	go func() {
		wg.Wait()
		close(results)
	}()

	// Consumidor
	suma := 0
	for r := range results {
		fmt.Printf("worker %d: %2d² = %3d\n", r.Worker, r.In, r.Out)
		suma += r.Out
	}
	fmt.Println("suma de cuadrados:", suma) // 385
}

Writing /content/go_ej1/02_pipeline/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run ./02_pipeline 2>&1 || echo "--> terminó con error (código $?)"

worker 1:  1² =   1
worker 3:  3² =   9
worker 2:  2² =   4
worker 1:  4² =  16
worker 3:  6² =  36
worker 2:  5² =  25
worker 3:  9² =  81
worker 1:  7² =  49
worker 2:  8² =  64
worker 3: 10² = 100
suma de cuadrados: 385


## 3) Timeout con `context.WithTimeout`
Dos casos: una tarea de 500 ms con timeout de 1 s (termina bien) y una de 3 s con timeout de 1 s (se cancela con `context deadline exceeded`).

In [ ]:
%%writefile /content/go_ej1/03_timeout/main.go
package main

import (
	"context"
	"fmt"
	"time"
)

func tarea(ctx context.Context, d time.Duration) error {
	select {
	case <-time.After(d):
		return nil // terminó a tiempo
	case <-ctx.Done():
		return ctx.Err() // timeout o cancelación
	}
}

func ejecutar(nombre string, duracion, timeout time.Duration) {
	ctx, cancel := context.WithTimeout(context.Background(), timeout)
	defer cancel() // libera recursos del contexto siempre

	inicio := time.Now()
	err := tarea(ctx, duracion)
	fmt.Printf("%-12s tarea=%v timeout=%v -> err=%v (tardó %v)\n",
		nombre, duracion, timeout, err, time.Since(inicio).Round(10*time.Millisecond))
}

func main() {
	ejecutar("rápida", 500*time.Millisecond, 1*time.Second)
	ejecutar("lenta", 3*time.Second, 1*time.Second)
}

Writing /content/go_ej1/03_timeout/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run ./03_timeout 2>&1 || echo "--> terminó con error (código $?)"

rápida       tarea=500ms timeout=1s -> err=<nil> (tardó 500ms)
lenta        tarea=3s timeout=1s -> err=context deadline exceeded (tardó 1s)


## 4) Deadlock
Un channel **sin buffer** exige que haya un emisor y un receptor al mismo tiempo. Acá `main` intenta enviar, pero nadie recibe.

In [ ]:
%%writefile /content/go_ej1/04_deadlock/main.go
package main

import "fmt"

func main() {
	ch := make(chan int) // sin buffer

	fmt.Println("main: voy a enviar...")
	ch <- 42 // main se bloquea acá esperando un receptor que nunca llega
	fmt.Println("main: nunca llego a esta línea")

	fmt.Println(<-ch)
}

Writing /content/go_ej1/04_deadlock/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run ./04_deadlock 2>&1 || echo "--> terminó con error (código $?)"

main: voy a enviar...
fatal error: all goroutines are asleep - deadlock!

goroutine 1 [chan send]:
main.main()
	/content/go_ej1/04_deadlock/main.go:9 +0x7e
exit status 2
--> terminó con error (código 1)


### Explicación
- **Quién espera:** la goroutine `main` está bloqueada en `ch <- 42`.
- **A quién espera:** a una goroutine que ejecute `<-ch`. Pero no existe ninguna: `main` es la única, y la recepción está *después* del envío, en la misma goroutine.
- Como **todas** las goroutines están dormidas, el runtime lo detecta y aborta con `fatal error: all goroutines are asleep - deadlock!`.

Variantes típicas del mismo problema: `wg.Wait()` sin `Done()` suficientes, o un `range` sobre un channel que nadie cierra.

Para corregirlo vamos a poner el receptor en otra goroutine (o usar un channel con buffer).

In [ ]:
%%writefile /content/go_ej1/04_deadlock/main.go
package main

import "fmt"

func main() {
	ch := make(chan int)

	go func() { ch <- 42 }() // el emisor ahora corre en otra goroutine

	fmt.Println("recibido:", <-ch) // main recibe -> se emparejan
}

Overwriting /content/go_ej1/04_deadlock/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run ./04_deadlock 2>&1 || echo "--> terminó con error (código $?)"

recibido: 42


## 5) Data race con `-race` y su corrección
100 goroutines hacen `contador++` sin sincronizar. `++` no es atómico (leer → sumar → escribir), así que se pisan entre sí: el resultado suele ser menor a 100 y el detector reporta `WARNING: DATA RACE`.

In [ ]:
%%writefile /content/go_ej1/05_carrera/main.go
package main

import (
	"fmt"
	"sync"
)

func main() {
	var wg sync.WaitGroup
	contador := 0

	for i := 0; i < 100; i++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			contador++ // ❌ CARRERA: acceso concurrente sin sincronizar
		}()
	}

	wg.Wait()
	fmt.Println("contador (esperado 100):", contador)
}

Writing /content/go_ej1/05_carrera/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run -race ./05_carrera 2>&1 || echo "--> terminó con error (código $?)"

Read at 0x00c00009c028 by goroutine 7:
  main.main.func1()
      /content/go_ej1/05_carrera/main.go:16 +0x84

Previous write at 0x00c00009c028 by goroutine 16:
  main.main.func1()
      /content/go_ej1/05_carrera/main.go:16 +0x96

Goroutine 7 (running) created at:
  main.main()
      /content/go_ej1/05_carrera/main.go:14 +0x7d

Goroutine 16 (finished) created at:
  main.main()
      /content/go_ej1/05_carrera/main.go:14 +0x7d
Write at 0x00c00009c028 by goroutine 69:
  main.main.func1()
      /content/go_ej1/05_carrera/main.go:16 +0x96

Previous write at 0x00c00009c028 by goroutine 44:
  main.main.func1()
      /content/go_ej1/05_carrera/main.go:16 +0x96

Goroutine 69 (running) created at:
  main.main()
      /content/go_ej1/05_carrera/main.go:14 +0x7d

Goroutine 44 (running) created at:
  main.main()
      /content/go_ej1/05_carrera/main.go:14 +0x7d
contador (esperado 100): 100
Found 2 data race(s)
exit status 66
--> terminó con error (código 1)


### Corrección con `sync.Mutex`
Alternativa: `sync/atomic` con `atomic.AddInt64`.

In [ ]:
%%writefile /content/go_ej1/05_carrera/main.go
package main

import (
	"fmt"
	"sync"
)

type Contador struct {
	mu sync.Mutex
	n  int
}

func (c *Contador) Inc() {
	c.mu.Lock()
	defer c.mu.Unlock()
	c.n++
}

func (c *Contador) Valor() int {
	c.mu.Lock()
	defer c.mu.Unlock()
	return c.n
}

func ContarConcurrente(goroutines int) int {
	var wg sync.WaitGroup
	c := &Contador{}
	for i := 0; i < goroutines; i++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			c.Inc() // ✅ protegido por mutex
		}()
	}
	wg.Wait()
	return c.Valor()
}

func main() {
	fmt.Println("contador (esperado 100):", ContarConcurrente(100))
}

Overwriting /content/go_ej1/05_carrera/main.go


In [ ]:
%%writefile /content/go_ej1/05_carrera/main_test.go
package main

import "testing"

func TestContarConcurrente(t *testing.T) {
	if got := ContarConcurrente(1000); got != 1000 {
		t.Fatalf("esperaba 1000, obtuve %d", got)
	}
}

Writing /content/go_ej1/05_carrera/main_test.go


In [ ]:
%%bash
cd /content/go_ej1
echo "=== go run -race ===";  go run -race ./05_carrera 2>&1 || echo "--> error (código $?)"
echo; echo "=== go test -race ==="; go test -race -count=1 ./05_carrera 2>&1 || echo "--> error (código $?)"

=== go run -race ===
contador (esperado 100): 100

=== go test -race ===
ok  	ejercicio1/05_carrera	1.016s


---
# Crawler simulado
- **Jobs** = URLs, enviadas por un channel.
- **Workers** "descargan" con `time.Sleep` (duración aleatoria).
- **Results** traen el status (200 / 404 / 500) o el error de cancelación.
- Todo se **cancela a los 2 segundos** con `context.WithTimeout`.


In [ ]:
%%writefile /content/go_ej1/06_crawler/main.go
package main

import (
	"context"
	"fmt"
	"math/rand"
	"sync"
	"time"
)

type Job struct {
	ID  int
	URL string
}

type Result struct {
	Job      Job
	Worker   int
	Status   int
	Duration time.Duration
	Err      error
}

var statuses = []int{200, 200, 200, 200, 404, 500}

// "Descarga" simulada: duerme entre 300 y 1500 ms, o aborta si se cancela el contexto.
func fetch(ctx context.Context, j Job) (int, error) {
	d := time.Duration(300+rand.Intn(1200)) * time.Millisecond
	select {
	case <-time.After(d):
		return statuses[rand.Intn(len(statuses))], nil
	case <-ctx.Done():
		return 0, ctx.Err()
	}
}

func worker(ctx context.Context, id int, jobs <-chan Job, results chan<- Result, wg *sync.WaitGroup) {
	defer wg.Done()
	for j := range jobs {
		inicio := time.Now()
		status, err := fetch(ctx, j)
		results <- Result{Job: j, Worker: id, Status: status, Duration: time.Since(inicio), Err: err}
		if err != nil {
			return // contexto cancelado: el worker termina
		}
	}
}

func main() {
	ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
	defer cancel()

	urls := make([]string, 12)
	for i := range urls {
		urls[i] = fmt.Sprintf("https://ejemplo.com/pagina/%d", i+1)
	}

	jobs := make(chan Job)
	results := make(chan Result)

	// Productor: envía URLs, pero deja de enviar si se cancela.
	go func() {
		defer close(jobs)
		for i, u := range urls {
			select {
			case jobs <- Job{ID: i + 1, URL: u}:
			case <-ctx.Done():
				return
			}
		}
	}()

	// Workers
	var wg sync.WaitGroup
	const nWorkers = 3
	for w := 1; w <= nWorkers; w++ {
		wg.Add(1)
		go worker(ctx, w, jobs, results, &wg)
	}
	go func() { wg.Wait(); close(results) }()

	// Consumidor
	ok, fallidos, cancelados := 0, 0, 0
	for r := range results {
		switch {
		case r.Err != nil:
			cancelados++
			fmt.Printf("[w%d] %-32s cancelado (%v)\n", r.Worker, r.Job.URL, r.Err)
		case r.Status == 200:
			ok++
			fmt.Printf("[w%d] %-32s %d en %v\n", r.Worker, r.Job.URL, r.Status, r.Duration.Round(time.Millisecond))
		default:
			fallidos++
			fmt.Printf("[w%d] %-32s %d en %v\n", r.Worker, r.Job.URL, r.Status, r.Duration.Round(time.Millisecond))
		}
	}

	procesadas := ok + fallidos + cancelados
	fmt.Printf("\nResumen: %d OK, %d con error HTTP, %d cancelados | %d URLs nunca se enviaron\n",
		ok, fallidos, cancelados, len(urls)-procesadas)
}

Overwriting /content/go_ej1/06_crawler/main.go


In [ ]:
%%bash
cd /content/go_ej1
go run -race ./06_crawler 2>&1 || echo "--> terminó con error (código $?)"

[w3] https://ejemplo.com/pagina/1     200 en 332ms
[w2] https://ejemplo.com/pagina/2     200 en 812ms
[w3] https://ejemplo.com/pagina/4     200 en 689ms
[w2] https://ejemplo.com/pagina/5     200 en 344ms
[w1] https://ejemplo.com/pagina/3     200 en 1.465s
[w2] https://ejemplo.com/pagina/7     404 en 726ms
[w2] https://ejemplo.com/pagina/9     cancelado (context deadline exceeded)
[w1] https://ejemplo.com/pagina/8     cancelado (context deadline exceeded)
[w3] https://ejemplo.com/pagina/6     cancelado (context deadline exceeded)

Resumen: 5 OK, 1 con error HTTP, 3 cancelados | 3 URLs nunca se enviaron
